# 05 — 학습 결과 읽기

**무엇을 확인하는가**

1. `runs/` 의 로그에서 지표를 뽑는다
2. LOCK 의 `interval` 항목과 맞춰본다
3. 하드웨어 기록이 있는지 확인한다

**첫 태그 범위 밖입니다.** `train/` 과 `experiments/` 는 자리만 잡아둔
상태입니다. 라벨 검증(00~03)이 끝난 뒤에 손대십시오.

## digest 와 interval

`digest` 는 순수 함수 출력이라 바이트가 같아야 하고, `interval` 은 난수
초기화와 GPU 커널 비결정성 때문에 구간으로만 같습니다.
**우열이 아니라 대상의 성질입니다.**

`manifests/hardware.txt` 가 비어 있으면 `interval` 기준값은 뜻이 없습니다.

## 0. 부트스트랩

In [ ]:
import sys
from pathlib import Path

# 노트북에서 저장소 루트를 import 경로에 넣습니다.
REPO = Path.cwd()
while not (REPO / "run.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("저장소 루트:", REPO)

from verify import read_text
from train import collect as collect_mod

RUNS = REPO / "runs"
print("runs/:", "있음" if RUNS.exists() else "없음 (아직 아무것도 돌리지 않았습니다)")

## 1. 실행 목록

`changes.txt` 를 반드시 확인하십시오. **무엇을 바꿔 돌렸는지 모르면 결과가
갈렸을 때 원인을 코드에서 찾게 됩니다.**

In [ ]:
if RUNS.exists():
    for path in sorted(p for p in RUNS.iterdir() if p.is_dir()):
        log = path / "log.txt"
        size = log.stat().st_size if log.exists() else 0
        print(f"  {path.name:40} log {size:>10} bytes")
else:
    print("(없음)")

## 2. 지표

상위 코드는 지표를 파일로 남기지 않습니다. wandb 로 보내고 stdout 에 찍을
뿐이라 로그를 파싱합니다 (`run_main.py:427-430`).

In [ ]:
records = []
if RUNS.exists():
    for path in sorted(p for p in RUNS.iterdir() if p.is_dir()):
        record = collect_mod.collect(path)
        records.append(record)
        print(f"\n[{record['run']}]")
        if record["metrics"]:
            width = max(len(k) for k in record["metrics"])
            for key, value in record["metrics"].items():
                print(f"  {key.ljust(width)}  {value}")
        else:
            print("  지표 없음 —", record["note"] or "로그가 없습니다")

## 3. 읽을 때 조심할 것

- **MAPE 는 비율입니다.** `utils/metrics.py:26` 이
  `mean(|(pred-true)/true|)` 를 돌려줍니다. 백분율이 아닙니다. 논문 표와
  비교할 때 100 을 곱해야 하는지 먼저 확인하십시오.
- **15%-Acc 는 `alpha_acc1`(=0.15) 입니다.** `run_main.py:125-126` 의 도움말
  문구가 값과 어긋나 있습니다 (`--alpha1` 의 help 가 "the 10 percent alpha").
  값과 출력 라벨은 맞으니 **도움말을 따라가지 마십시오.**
- `Test Seen` 과 `Test Unseen` 이 따로 찍힙니다. 논문 표가 어느 쪽인지
  확인하십시오.

## 4. 하드웨어 기록

`interval` 항목은 이 기록 없이는 뜻이 없습니다.

In [ ]:
hardware = REPO / "manifests" / "hardware.txt"
text = read_text(hardware)
unset = text.count("(미정)")
print(f"manifests/hardware.txt — (미정) {unset}개")
if unset:
    print("\n비어 있습니다. 학습을 처음 돌린 사람이 채웁니다.")
    print("GPU · 드라이버 · CUDA · torch · 시드 · cudnn 설정이 바뀌면")
    print("interval 기준값을 다시 재야 합니다.")

## 5. LOCK 의 interval 항목

`digest` 항목과 달리 `lock-init` 이 채우지 않습니다. 사람이 구간을 정해
`LOCK.md` 에 직접 적습니다. 한 번 돌린 값을 그대로 기준으로 삼지 말고,
**몇 번 돌려 폭을 보고** 적으십시오.

In [ ]:
from verify import lock

result = lock.check()
for row in result["rows"]:
    if row["kind"] == "interval":
        print(f"  {row['item']:36} {row['expected']:20} {row['verdict']}")
        print(f"    {row['note']}")